# Create Approver Experience Builder App

Clone and configure an Experience Builder app for the approver workflow.

## Workflow
1. Clone the Experience Builder template
2. Update data sources to use our Approver web map
3. Configure title and sharing
4. Save to registry

In [7]:
from arcgis.gis import GIS
import json
from pathlib import Path
import sys
import getpass
import warnings

# Suppress resource and SSL warnings
warnings.filterwarnings('ignore', category=ResourceWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Add repo root to path
_repo_root = Path.cwd().parent
sys.path.insert(0, str(_repo_root))

import views

In [8]:
# Configuration
STAGING = True

# Experience Builder template ID
# This is the app created by a colleague that we'll use as a template
EXPERIENCE_TEMPLATE_ID = 'a535042415e749e790c7f47c275340c2'

# Title suffix and registry path
_suffix = " - STAGING" if STAGING else ""
_reg_file = _repo_root / ("item_registry_staging.json" if STAGING else "item_registry.json")

print(f"Mode: {'STAGING' if STAGING else 'PRODUCTION'}")
print(f"Registry: {_reg_file.name}")
print(f"Template: {EXPERIENCE_TEMPLATE_ID}")

Mode: STAGING
Registry: item_registry_staging.json
Template: a535042415e749e790c7f47c275340c2


In [9]:
# Connect to ArcGIS Online
username = input("ArcGIS Online username: ")
password = getpass.getpass("Password: ")

gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Connected as: {gis.users.me.username}")
print(f"Organization: {gis.properties.name}")

Setting `verify_cert` to False is a security risk, use at your own risk.


Connected as: ralouta.aiddev
Organization: Esri Aid & Development Team


## Load Registry

Load the registry to get the Approver web map ID.

In [10]:
# Load registry
registry = views.load_registry(_reg_file)

# Get the Approver web map
if 'web_maps' not in registry or 'approver' not in registry['web_maps']:
    raise ValueError(
        "Approver web map not found in registry. "
        "Run manage-views-and-webmaps.ipynb first to create it."
    )

approver_webmap_id = registry['web_maps']['approver']['item_id']
approver_view_id = registry['views']['approver']['item_id']

# Get the folder from the registry (saved when base layer was created)
TARGET_FOLDER_ID = registry.get('base_layer', {}).get('folder')
TARGET_FOLDER_NAME = registry.get('base_layer', {}).get('folder_name', '(root)')

print(f"Approver web map: {approver_webmap_id}")
print(f"Approver view layer: {approver_view_id}")
print(f"Target folder: {TARGET_FOLDER_NAME}")
if TARGET_FOLDER_ID:
    print(f"  Folder ID: {TARGET_FOLDER_ID}")

# Verify they exist
approver_wm = gis.content.get(approver_webmap_id)
approver_view = gis.content.get(approver_view_id)

print(f"\nWeb map title: {approver_wm.title}")
print(f"View layer title: {approver_view.title}")

Approver web map: 3ac7652e0aee46d0a774c00fb3c6b46a
Approver view layer: 55318c40af0c477abeef5d611cc94d20
Target folder: Plant Identification - Staging
  Folder ID: 1657ba943f4545198c54797359dbb665

Web map title: Plant Identificaion - AI - Approver Map - STAGING
View layer title: Plant Identificaion - AI - Approver - STAGING


## Clone Experience Builder Template

In [11]:
import importlib, tools.exb; importlib.reload(tools.exb)
from tools.exb import update_datasources, sync_config_resources

# Get the template and its config to find the old web map ID
template = gis.content.get(EXPERIENCE_TEMPLATE_ID)
print(f"Template: {template.title}")
print(f"Type: {template.type}")
print(f"Owner: {template.owner}")

# Find the web map ID used by the template
template_config = template.get_data()
wm_ds = next(
    (ds for ds in template_config.get('dataSources', {}).values() if ds.get('type') == 'WEB_MAP'),
    None,
)
config_wm_id = wm_ds.get('itemId') if wm_ds else None
print(f"\nTemplate web map: {config_wm_id}")
print(f"Target web map:   {approver_webmap_id}")

# Build item_mapping so clone_items remaps the web map
# instead of cloning it as a dependency
item_mapping = {}
if config_wm_id and config_wm_id != approver_webmap_id:
    item_mapping[config_wm_id] = approver_webmap_id
    print(f"Mapping: {config_wm_id} → {approver_webmap_id}")

# Also map any feature layer item IDs referenced in children
if wm_ds:
    for child in wm_ds.get('childDataSourceJsons', {}).values():
        old_fl_id = child.get('itemId', '')
        if old_fl_id and old_fl_id not in item_mapping:
            # Map old feature layer items to our view layers
            # (clone_items will skip cloning them)
            pass  # only the web map needs mapping; layers resolve from it

new_title = f"Plant Identification - Approver App{_suffix}"
clone_folder = TARGET_FOLDER_NAME if TARGET_FOLDER_NAME != "(root)" else None
target_folder = TARGET_FOLDER_NAME if TARGET_FOLDER_NAME != "(root)" else "/"

# Clone using the proper ExB clone mechanism
cloned = gis.content.clone_items(
    items=[template],
    folder=clone_folder,
    item_mapping=item_mapping,
    copy_data=False,
    search_existing_items=False,
)

if not cloned:
    raise RuntimeError("clone_items returned empty list")

cloned_app = cloned[0]

# Ensure the app ends up in the same folder as the source content
if TARGET_FOLDER_ID:
    if cloned_app.ownerFolder != TARGET_FOLDER_ID:
        move_result = cloned_app.move(TARGET_FOLDER_NAME)
        print(f"✓ App moved to folder: {TARGET_FOLDER_NAME} ({move_result})")
    else:
        print(f"✓ App already in folder: {TARGET_FOLDER_NAME}")
else:
    if cloned_app.ownerFolder:
        move_result = cloned_app.move("/")
        print(f"✓ App moved to folder: {TARGET_FOLDER_NAME} ({move_result})")
    else:
        print("✓ App already in folder: (root)")

# Update title and snippet
cloned_app.update(item_properties={
    'title': new_title,
    'tags': 'plant identification, approver, experience builder, field collection',
    'snippet': f"Experience Builder app for reviewing and approving plant identification submissions{_suffix}",
})

# Fix widget data source references in both published and draft configs
config = cloned_app.get_data()
config = update_datasources(config, gis, approver_webmap_id)
cloned_app.update(data=json.dumps(config))
draft_updates = sync_config_resources(cloned_app, config)
print("✓ Widget data sources remapped")
print(f"✓ Updated {draft_updates} draft config resource(s)")

app_url = f"https://experience.arcgis.com/experience/{cloned_app.id}"

print(f"\n✓ Experience Builder app cloned:")
print(f"  Title: {cloned_app.title}")
print(f"  ID: {cloned_app.id}")
print(f"  URL: {app_url}")
print(f"  Builder: https://experience.arcgis.com/builder/?id={cloned_app.id}")


Template: Plant Identification - Approver App
Type: Web Experience
Owner: ralouta.aiddev

Template web map: e45b1f56d77c469199c6ab02d66e1e19
Target web map:   3ac7652e0aee46d0a774c00fb3c6b46a
Mapping: e45b1f56d77c469199c6ab02d66e1e19 → 3ac7652e0aee46d0a774c00fb3c6b46a
✓ App already in folder: Plant Identification - Staging
✓ Widget data sources remapped
✓ Updated 1 draft config resource(s)

✓ Experience Builder app cloned:
  Title: Plant Identification - Approver App - STAGING
  ID: a54bf9d3366440da8ac312ac5e8803cc
  URL: https://experience.arcgis.com/experience/a54bf9d3366440da8ac312ac5e8803cc
  Builder: https://experience.arcgis.com/builder/?id=a54bf9d3366440da8ac312ac5e8803cc


## Configure Icons

Drop images into the repo's `icons/` folder, then run the cell below. Each detected image resource on the cloned app is shown with its current preview and a dropdown to choose a replacement. Click **Apply selected replacements** to update the app.

In [ ]:
# Icon picker — lists every image resource on the cloned app and lets
# you swap any of them with a file from the repo's ``icons/`` folder.
import importlib, tools.exb_icons
importlib.reload(tools.exb_icons)
from tools.exb_icons import (
    list_icon_resources, list_local_icons,
    replace_icon, download_icon_preview,
)
from pathlib import Path
import tempfile
import ipywidgets as w
from IPython.display import display

_icons_dir = _repo_root / "icons"
_local_icons = list_local_icons(_icons_dir)
_resources = list_icon_resources(cloned_app)

if not _resources:
    print("No image resources found on the app.")
elif not _local_icons:
    print(f"No icon files in {_icons_dir}. Drop PNG/JPG/SVG files there and rerun.")
else:
    _preview_dir = Path(tempfile.mkdtemp(prefix="exb_icon_preview_"))
    _dropdown_options = [("— keep current —", "")] + [
        (p.name, str(p)) for p in _local_icons
    ]

    _rows = []
    _selectors: dict[str, w.Dropdown] = {}

    for entry in _resources:
        res_path = entry["resource"]
        try:
            preview_path = download_icon_preview(cloned_app, res_path, _preview_dir)
            img_bytes = preview_path.read_bytes()
            img = w.Image(value=img_bytes, format=preview_path.suffix.lstrip(".") or "png",
                          width=64, height=64)
        except Exception as exc:
            img = w.HTML(value=f"<i>(no preview: {exc})</i>")

        dd = w.Dropdown(options=_dropdown_options, value="",
                        description="replace:", layout=w.Layout(width="320px"))
        _selectors[res_path] = dd

        label = w.HTML(value=f"<b>{entry['label']}</b><br/><code>{res_path}</code>")
        _rows.append(w.HBox([img, label, dd],
                            layout=w.Layout(align_items="center", gap="16px", margin="6px 0")))

    apply_btn = w.Button(description="Apply selected replacements",
                         button_style="primary", icon="check")
    status = w.Output()

    def _on_apply(_):
        with status:
            status.clear_output()
            updated = 0
            for res_path, dd in _selectors.items():
                choice = dd.value
                if not choice:
                    continue
                try:
                    replace_icon(cloned_app, res_path, Path(choice))
                    print(f"✓ {res_path}  ←  {Path(choice).name}")
                    updated += 1
                except Exception as exc:
                    print(f"✗ {res_path}: {exc}")
            if updated == 0:
                print("No selections to apply.")
            else:
                print(f"\nDone — {updated} resource(s) updated. Hard-refresh the ExB tab to see the new icons.")

    apply_btn.on_click(_on_apply)
    display(w.VBox(_rows + [apply_btn, status]))


## Share the App

In [13]:
# Share with organization (approvers need access)
share_result = cloned_app.share(org=True)
print(f"Shared with organization: {share_result}")

print(f"\n✓ App created successfully!")
print(f"  Title: {cloned_app.title}")
print(f"  ID: {cloned_app.id}")
print(f"  URL: {app_url}")

/Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esriaiddev.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Shared with organization: {'notSharedWith': [], 'itemId': 'a54bf9d3366440da8ac312ac5e8803cc'}

✓ App created successfully!
  Title: Plant Identification - Approver App - STAGING
  ID: a54bf9d3366440da8ac312ac5e8803cc
  URL: https://experience.arcgis.com/experience/a54bf9d3366440da8ac312ac5e8803cc


## Save to Registry

In [14]:
# Add to registry
if 'web_apps' not in registry:
    registry['web_apps'] = {}

registry['web_apps']['approver'] = {
    'item_id': cloned_app.id,
    'title': cloned_app.title,
    'url': app_url,
    'type': cloned_app.type,
    'template_id': EXPERIENCE_TEMPLATE_ID
}

# Save registry
views.save_registry(_reg_file, registry)
print(f"✓ Saved to registry: {_reg_file.name}")
print(f"\nRegistry entry:")
print(json.dumps(registry['web_apps']['approver'], indent=2))

✓ Saved to registry: item_registry_staging.json

Registry entry:
{
  "item_id": "a54bf9d3366440da8ac312ac5e8803cc",
  "title": "Plant Identification - Approver App - STAGING",
  "url": "https://experience.arcgis.com/experience/a54bf9d3366440da8ac312ac5e8803cc",
  "type": "Web Experience",
  "template_id": "a535042415e749e790c7f47c275340c2"
}
